In [3]:
'''
This block sets some parameters for the three-states Potts and fixes a fundamental domain for the Kac labels (called Epp)
Note that we work with the chiral algebra for the minimal model
'''

# This will be the (p,p'=pp) minimal model
p = 6  # p
pp = 5  # p'
# Various constant parameters of the minimal model
N = 2 * p * pp
num_prim = (p-1)*(pp-1) // 2  # Number of primaries

c = 1 - 6 * (p - pp)**2 / (p * pp) # Central charge

# Index set
####### This needs to be changed for each minimal model ########
Epp = [(1,1), (1,2), (1,3), (1,4), (1,5), (2,2), (2,3), (2,4), (2,5), (3,5)] # By (r,s); this is for three states potts
# The conformal weights
cdim = [ ((p*r-pp*s)**2 - (p-pp)**2) / (2*N) for r,s in Epp ]

# Modular S-matrix
S = matrix([ [ QQbar(4*sqrt( 1/QQ(N) )*(-1)**(1+s*u+r*v)*sin(pi*QQ(p)/QQ(pp)*r*u)*sin(pi*QQ(pp)/QQ(p)*s*v)).radical_expression() for r,s in Epp ] for u,v in Epp ])

# Mass matrix
M = matrix( 10, 10 )
for r in {1,2}:
    rs = (1,1) if r==1 else (pp-r,p-1)
    M[Epp.index(rs),Epp.index(rs)] = 1
    M[Epp.index(rs),Epp.index((r,5))] = 1
    M[Epp.index((r,5)),Epp.index(rs)] = 1
    M[Epp.index((r,5)),Epp.index((r,5))] = 1
    M[Epp.index((r,3)),Epp.index((r,3))] = 2

In [63]:
print(S.is_symmetric())
print(S.is_unitary())
print(S*M*S == M)

True
True
True


In [ ]:
import itertools as itr


i = 0
L = matrix(SR,10,10)
for a,b in itr.product(range(10),repeat=2):
    if M[a,b]:
        L[a,b] = M[a,b]*var(f'L{i}')
        i+=1  
        
N = matrix( mathematica(S*L*S).Simplify().sage() )
ind = [(1,1),(2,1),(5,1),(6,1),(1,2),(2,2),(4,2),(5,2),(6,2),(8,2)]
mat = matrix( [[N[j,k].diff(var(f'L{i}')) for i in range(10)] for j,k in ind] )


mat_inv = matrix( mathematica(mat.inverse()).Simplify().sage() )
L_vec = mat_inv*vector([var(f'n{i}') for i in range(10)])

constraints = set()
for j,k in itr.product(range(10),repeat=2):
    cons = N[j,k].subs([var(f'L{i}')==L_vec[i] for i in range(10)])
    cons = cons.simplify(algorithm = 'giac')
    constraints.add(cons)

constraints = constraints - {var(f'n{i}') for i in range(10)}


In [23]:
constraints

{1/2*n1 + 1/2*n3,
 1/2*n1,
 1/2*n5 - 1/2*n6,
 1/2*n7,
 1/2*n8 - 1/2*n9,
 1/2*n5 - 1/2*n6 + 1/2*n8 - 1/2*n9,
 n1 + n3,
 n4 + n7,
 1/2*n3,
 1/2*n4,
 n5 + n8,
 n0 + n2,
 1/2*n4 + 1/2*n7,
 n6 + n9}

In [84]:
# Solved the constraints by hand and got the following 12 element generating set
sols = [(1,0,0,0,0,0,0,0,0,0),\
        (0,2,0,0,0,0,0,0,0,0),\
        (0,0,1,0,0,0,0,0,0,0),\
        (0,0,0,2,0,0,0,0,0,0),\
        (0,0,0,0,2,0,0,0,0,0),\
        (0,0,0,0,0,2,0,0,0,0),\
        (0,0,0,0,0,1,1,0,0,0),\
        (0,0,0,0,0,0,0,2,0,0),\
        (0,0,0,0,0,0,0,0,2,0),\
        (0,0,0,0,0,0,0,0,1,1)]

all_defects = []
for sol in sols:
    L_vec = mat_inv*vector(sol)
    L_vec = vector( [L_vec[i].simplify(algorithm='giac') for i in range(10)] )
    all_defects.append(L_vec)
    i = 0
    L = matrix(SR,10,10)
    for a,b in itr.product(range(10),repeat=2):
        if M[a,b]:
            L[a,b] = L_vec[i]
            i+=1
    print(L.is_symmetric())
    print(L[0,0])
    
    

True
1
False
sqrt(3)
True
1/2*sqrt(5) + 1/2
False
1/2*sqrt(15) + 1/2*sqrt(3)
False
sqrt(3)
True
1
True
1
False
1/2*sqrt(15) + 1/2*sqrt(3)
True
1/2*sqrt(5) + 1/2
True
1/2*sqrt(5) + 1/2


In [92]:
all_defects

[(1, -1, 0, -1, 1, 0, 1, -1, -1, 1),
 (sqrt(3), -sqrt(3), 0, sqrt(3), -sqrt(3), 0, sqrt(3), -sqrt(3), sqrt(3), -sqrt(3)),
 (1/2*sqrt(5) + 1/2, -1/2*sqrt(5) - 1/2, 0, -1/2*sqrt(5) - 1/2, 1/2*sqrt(5) + 1/2, 0, -1/2*sqrt(5) + 1/2, 1/2*sqrt(5) - 1/2, 1/2*sqrt(5) - 1/2, -1/2*sqrt(5) + 1/2),
 (1/2*sqrt(15) + 1/2*sqrt(3), -1/2*sqrt(15) - 1/2*sqrt(3), 0, 1/2*sqrt(15) + 1/2*sqrt(3), -1/2*sqrt(15) - 1/2*sqrt(3), 0, -1/2*sqrt(15) + 1/2*sqrt(3), 1/2*sqrt(15) - 1/2*sqrt(3), -1/2*sqrt(15) + 1/2*sqrt(3), 1/2*sqrt(15) - 1/2*sqrt(3)),
 (sqrt(3), sqrt(3), 0, -sqrt(3), -sqrt(3), 0, sqrt(3), sqrt(3), -sqrt(3), -sqrt(3)),
 (1, 1, 1, 1, 1, 1, 1, 1, 1, 1),
 (1, 1, -1/2, 1, 1, -1/2, 1, 1, 1, 1),
 (1/2*sqrt(15) + 1/2*sqrt(3), 1/2*sqrt(15) + 1/2*sqrt(3), 0, -1/2*sqrt(15) - 1/2*sqrt(3), -1/2*sqrt(15) - 1/2*sqrt(3), 0, -1/2*sqrt(15) + 1/2*sqrt(3), -1/2*sqrt(15) + 1/2*sqrt(3), 1/2*sqrt(15) - 1/2*sqrt(3), 1/2*sqrt(15) - 1/2*sqrt(3)),
 (1/2*sqrt(5) + 1/2, 1/2*sqrt(5) + 1/2, 1/2*sqrt(5) + 1/2, 1/2*sqrt(5) + 1/2, 1/2*

In [122]:
def vector_decomp(vec, parts):
    dim = len(vec)
    # Base case: exactly decomposed n.
    if vec == vector(QQbar, dim):
        return [[]]
    
    res = [] 
    
    # Try each square starting from index `start`
    for p in parts:
        if all(p[i] <= vec[i] for i in range(dim)):
            # For each decomposition of the remainder, prepend the current square.
            for sub in vector_decomp(vec - p, parts):
                res.append([ p ] + sub)
    
    return res

all_defects = sorted( [ vector(QQbar,d) for d in all_defects ] )
new_defect = vector(all_defects[3][i]*all_defects[6][i] for i in range(10))

vector_decomp(new_defect, all_defects)

[[(2.802517076888147?, 2.802517076888147?, 0, -2.802517076888147?, -2.802517076888147?, 0, -1.070466269319270?, -1.070466269319270?, 1.070466269319270?, 1.070466269319270?)]]

In [115]:
all_defects

[(1, -1, 0, -1, 1, 0, 1, -1, -1, 1),
 (1, 1, -1/2, 1, 1, -1/2, 1, 1, 1, 1),
 (1, 1, 1, 1, 1, 1, 1, 1, 1, 1),
 (1.618033988749895?, -1.618033988749895?, 0, -1.618033988749895?, 1.618033988749895?, 0, -0.618033988749895?, 0.618033988749895?, 0.618033988749895?, -0.618033988749895?),
 (1.618033988749895?, 1.618033988749895?, -0.8090169943749474?, 1.618033988749895?, 1.618033988749895?, 0.3090169943749474?, -0.618033988749895?, -0.618033988749895?, -0.618033988749895?, -0.618033988749895?),
 (1.618033988749895?, 1.618033988749895?, 1.618033988749895?, 1.618033988749895?, 1.618033988749895?, -0.618033988749895?, -0.618033988749895?, -0.618033988749895?, -0.618033988749895?, -0.618033988749895?),
 (1.732050807568878?, -1.732050807568878?, 0, 1.732050807568878?, -1.732050807568878?, 0, 1.732050807568878?, -1.732050807568878?, 1.732050807568878?, -1.732050807568878?),
 (1.732050807568878?, 1.732050807568878?, 0, -1.732050807568878?, -1.732050807568878?, 0, 1.732050807568878?, 1.732050807568878

In [65]:
P = Polyhedron(ieqs = [(0,1,0,0),(0,0,1,0),(0,0,0,1)],base_ring=ZZ)
dir(P)

['Hrep_generator',
 'Hrepresentation',
 'Hrepresentation_space',
 'Hrepresentation_str',
 'Hstar_function',
 'Vrep_generator',
 'Vrepresentation',
 'Vrepresentation_space',
 '_Hrepresentation',
 '_Hstar_function_normaliz',
 '_Vrepresentation',
 '__add__',
 '__and__',
 '__bool__',
 '__class__',
 '__contains__',
 '__copy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__floordiv__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getmetaclass__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__matmul__',
 '__mod__',
 '__module__',
 '__mul__',
 '__ne__',
 '__neg__',
 '__new__',
 '__pari__',
 '__pos__',
 '__pow__',
 '__pyx_vtable__',
 '__radd__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__rfloordiv__',
 '__rmatmul__',
 '__rmod__',
 '__rmul__',
 '__rpow__',
 '__rsub__',
 '__rtruediv__',
 '__rxor__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__slots__',
 '__static_attributes__',


In [ ]:
A = matrix(ZZ,[[2,3,4],[-4,4,-1],[0,2,1]])
D,U,V = A.smith_form()

U.inverse()*D*V.inverse()

[ 2  3  4]
[-4  4 -1]
[ 0  2  1]

In [28]:
dhsw_snf?

Signature:      dhsw_snf(mat, verbose=False)
Docstring:     
Preprocess a matrix using the "Elimination algorithm" described by
Dumas et al. [DHSW2003], and then call "elementary_divisors" on the
resulting (smaller) matrix.

Note:

  'snf' stands for 'Smith Normal Form'.

INPUT:

* "mat" -- integer matrix, either sparse or dense

(They use the transpose of the matrix considered here, so they use
rows instead of columns.)

ALGORITHM:

Go through "mat" one column at a time.  For each column, add multiples
of previous columns to it until either

* it's zero, in which case it should be deleted.

* its first nonzero entry is 1 or -1, in which case it should be kept.

* its first nonzero entry is something else, in which case it is
  deferred until the second pass.

Then do a second pass on the deferred columns.

At this point, the columns with 1 or -1 in the first entry contribute
to the rank of the matrix, and these can be counted and then deleted
(after using the 1 or -1 entry to clear ou

In [31]:
matrix(ZZ,[[2,3,4],[-4,4,-1],[0,2,1]]).elementary_divisors()

[1, 1, 8]

In [33]:
help(dhsw_snf)

Help on function dhsw_snf in module sage.homology.matrix_utils:

dhsw_snf(mat, verbose=False)
    Preprocess a matrix using the "Elimination algorithm" described by
    Dumas et al. [DHSW2003]_, and then call ``elementary_divisors`` on the
    resulting (smaller) matrix.

    .. NOTE::

        'snf' stands for 'Smith Normal Form'.

    INPUT:

    - ``mat`` -- integer matrix, either sparse or dense

    (They use the transpose of the matrix considered here, so they use
    rows instead of columns.)

    ALGORITHM:

    Go through ``mat`` one column at a time.  For each
    column, add multiples of previous columns to it until either

    - it's zero, in which case it should be deleted.
    - its first nonzero entry is 1 or -1, in which case it should be kept.
    - its first nonzero entry is something else, in which case it is
      deferred until the second pass.

    Then do a second pass on the deferred columns.

    At this point, the columns with 1 or -1 in the first entry
    co

In [38]:
mat_inv

[                           1                  1/2*sqrt(3)            1/2*sqrt(5) + 1/2  1/4*sqrt(3/5)*(sqrt(5) + 5)                  1/2*sqrt(3)                          1/2                          1/2  1/4*sqrt(3/5)*(sqrt(5) + 5)            1/4*sqrt(5) + 1/4            1/4*sqrt(5) + 1/4]
[                          -1                 -1/2*sqrt(3)           -1/2*sqrt(5) - 1/2 -1/4*sqrt(3/5)*(sqrt(5) + 5)                  1/2*sqrt(3)                          1/2                          1/2  1/4*sqrt(3/5)*(sqrt(5) + 5)            1/4*sqrt(5) + 1/4            1/4*sqrt(5) + 1/4]
[                           0                            0                            0                            0                            0                          1/2                           -1                            0            1/4*sqrt(5) + 1/4           -1/2*sqrt(5) - 1/2]
[                          -1                  1/2*sqrt(3)           -1/2*sqrt(5) - 1/2  1/4*sqrt(3/5)*(sqrt(5) + 5)        

In [ ]:
s1 = QQbar(2*sin(pi/5)/sqrt(15)).radical_expression()
s2 = QQbar(2*sin(2*pi/5)/sqrt(15)).radical_expression()
S = matrix([[s1, s2, 2*s1, 2*s2],\
            [s2, -s1, 2*s2, -2*s1],\
            [s1, s2, -s1, -s2],\
            [s2, -s1, -s2, s1]])
            
(S**2).simplify_full()

[1 0 0 0]
[0 1 0 0]
[0 0 1 0]
[0 0 0 1]

In [7]:
S.is_symmetric()

False

In [4]:
(S.T*diagonal_matrix([1,1,2,2])*S).simplify_full()

[1 0 0 0]
[0 1 0 0]
[0 0 2 0]
[0 0 0 2]

In [27]:
var('a,b,c,d')
n = ( S*diagonal_matrix([a,b,c,d])*S.T ).simplify_full()

In [48]:
n

[1/30*b*(sqrt(5) + 5) + 2/15*d*(sqrt(5) + 5) - 1/30*a*(sqrt(5) - 5) - 2/15*c*(sqrt(5) - 5)                             1/30*(a - b + 4*c - 4*d)*sqrt(sqrt(5) + 5)*sqrt(-sqrt(5) + 5) 1/30*b*(sqrt(5) + 5) - 1/15*d*(sqrt(5) + 5) - 1/30*a*(sqrt(5) - 5) + 1/15*c*(sqrt(5) - 5)                             1/30*(a - b - 2*c + 2*d)*sqrt(sqrt(5) + 5)*sqrt(-sqrt(5) + 5)]
[                            1/30*(a - b + 4*c - 4*d)*sqrt(sqrt(5) + 5)*sqrt(-sqrt(5) + 5) 1/30*a*(sqrt(5) + 5) + 2/15*c*(sqrt(5) + 5) - 1/30*b*(sqrt(5) - 5) - 2/15*d*(sqrt(5) - 5)                             1/30*(a - b - 2*c + 2*d)*sqrt(sqrt(5) + 5)*sqrt(-sqrt(5) + 5) 1/30*a*(sqrt(5) + 5) - 1/15*c*(sqrt(5) + 5) - 1/30*b*(sqrt(5) - 5) + 1/15*d*(sqrt(5) - 5)]
[1/30*b*(sqrt(5) + 5) - 1/15*d*(sqrt(5) + 5) - 1/30*a*(sqrt(5) - 5) + 1/15*c*(sqrt(5) - 5)                             1/30*(a - b - 2*c + 2*d)*sqrt(sqrt(5) + 5)*sqrt(-sqrt(5) + 5) 1/30*b*(sqrt(5) + 5) + 1/30*d*(sqrt(5) + 5) - 1/30*a*(sqrt(5) - 5) - 1/30*c*(sqrt(5) - 5)      

In [51]:
var('n1,n2,n3,n4')
sol = solve([n1 == n[0,0], n2==n[1,1],n3==n[2,2], n4==n[3,3]],[a,b,c,d])[0]

In [68]:
import itertools as itr

eqns = set()
for i,j in itr.product(range(4),repeat=2):
    this_guy = n[i,j].subs(sol).simplify(algorithm='giac')
    if this_guy not in {n1,n2,n3,n4}:
        eqns.add(this_guy)
        print(this_guy)

-n1 + n2
-n1 + 2*n3
n1 - n2 - 2*n3 + 2*n4
-n1 + n2
n1 - n2 - 2*n3 + 2*n4
-n2 + 2*n4
-n1 + 2*n3
n1 - n2 - 2*n3 + 2*n4
-n3 + n4
n1 - n2 - 2*n3 + 2*n4
-n2 + 2*n4
-n3 + n4


In [72]:
eqns

{n1 - n2 - 2*n3 + 2*n4, -n1 + n2, -n1 + 2*n3, -n2 + 2*n4, -n3 + n4}

In [71]:
solve([eq.subs(n2==1)==0 for eq in eqns],[n1,n3,n4])

[[n1 == 1, n3 == (1/2), n4 == (1/2)]]

[n2 - 1 == 0,
 2*n3 - 1 == 0,
 -n2 - 2*n3 + 2*n4 + 1 == 0,
 n2 - 1 == 0,
 n2 == 0,
 -n2 - 2*n3 + 2*n4 + 1 == 0,
 -n2 + 2*n4 == 0,
 2*n3 - 1 == 0,
 -n2 - 2*n3 + 2*n4 + 1 == 0,
 n3 == 0,
 -n3 + n4 == 0,
 -n2 - 2*n3 + 2*n4 + 1 == 0,
 -n2 + 2*n4 == 0,
 -n3 + n4 == 0,
 n4 == 0]